<div style="text-align: center;">
    <h1 style="color: #FF6347;">Self-Guided Lab: Retrieval-Augmented Generation (RAGs)</h1>
</div>

<h1 style="color: #FF6347;">Data Storage & Retrieval</h1>


In [4]:
import langchain
print(langchain.__version__)


1.3.0


In [6]:
import os
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_openai import OpenAIEmbeddings, ChatOpenAI
from langchain_chroma import Chroma
import warnings
warnings.filterwarnings('ignore')


<h3 style="color: #FF8C00;">Loading the Documents</h3>

In [7]:
# File path for the document

file_path = "../ai-for-everyone.pdf"

<h3 style="color: #FF8C00;">Documents into pages</h3>



In [9]:
# Load and split the document
loader = PyPDFLoader(file_path)
pages = loader.load_and_split()
len(pages)

297

<h3 style="color: #FF8C00;">Pages into Chunks</h3>

In [24]:
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=700,
    chunk_overlap=100
)
chunks = text_splitter.split_documents(pages)

len(chunks)

1501

<h2 style="color: #FF8C00;">Embeddings</h2>


In [25]:
from dotenv import load_dotenv, find_dotenv
import os

In [26]:
load_dotenv()

True

In [27]:
api_key = os.getenv("OPENAI_API_KEY")
embeddings = OpenAIEmbeddings(model="text-embedding-3-large")

<h2 style="color: #FF8C00;">ChromaDB</h2>


In [15]:
db = Chroma.from_documents(chunks, embeddings, persist_directory="chromadb_lab/chroma_db_LAB")
print("ChromaDB created with document embeddings.")

ChromaDB created with document embeddings.


<h1 style="color: #FF6347;">Retrieving Documents</h1>


### Exercice1: Write a user question that someone might ask about your book’s topic or content.

In [47]:
user_question = "Mention three main aspects of how AI can influence current political systems." # User question
retrieved_docs = db.similarity_search(user_question, k=10) # k is the number of documents to retrieve

In [48]:
# Display top results
for i, doc in enumerate(retrieved_docs[:3]): # Display top 3 results
    print(f"Document {i+1}:\n{doc.page_content[0:1000]}") # Display content

Document 1:
humans as workers, the idea of a universal basic income will be put back on the 
table. This would trigger a series of fundamental philosophical but also reli -
gious debates: questions such as the purpose of life, how to feel useful and what 
to strive for, are some issues for which society would have to find answers. Eth-
ics and education will play an important role in order to tackle these societal 
challenges and questions (Kaplan 2020a). 
About Democracy and Freedom
Finally, AI progress could represent nothing less than a danger to peace and 
democracy (Kaplan 2020b). There are at least two ways in which artificial  
intelligence might constitute a threat to democracy and its mechanisms,  
endangering the peaceful coexistence of humans and machines: supervision 
and manipulation.
Using the example of China, we will provide an illustration as to how far the 
possibilities of artificial intelligence reach with respect to control and super -
Document 2:
ment of AI across

<h2 style="color: #FF8C00;">Preparing Content for GenAI</h2>

In [49]:
def _get_document_prompt(docs):
    prompt = "\n"
    for doc in docs:
        prompt += "\nContent:\n"
        prompt += doc.page_content + "\n\n"
    return prompt

In [50]:
# Generate a formatted context from the retrieved documents
formatted_context = _get_document_prompt(retrieved_docs)
print("Context formatted for GPT model.")

Context formatted for GPT model.


<h2 style="color: #FF8C00;">ChatBot Architecture</h2>

### Exercice2: Write a prompt that is relevant and tailored to the content and style of your book.

In [62]:
prompt = f"""
## System role

I am an analytical assistant chatbot, in charge of providing context to the user's question based on the textbook provided. 


The user asks: 
"{user_question}"


Here are the most relevant ideas from the authors: 
'''
{formatted_context}

'''
## GUIDELINES: 
1. **Accuracy** 
- Use only the content provided to answer the question. 
- When the answer is not found, state: "The provided context does not contain information related to the user's question."
2. **Transparency**:
   - Reference the book's chapter title, author and page numbers when providing information.
   - Do not speculate or provide opinions.

3. **Clarity**:
   - Use simple, professional, and concise language.
   - Format your response in Markdown for readability.

## TASK
1. Summarize user question in simpler words. 
2. Decide which retrieved text chunks directly apply. .
3. Combine those chunks into an outline.
4. Draft a single coherent answer. 
5. Show the most relevant quotes from the authors, each with the page number where it can be found. 

## RESPONSE FORMAT
'''
[Summarize user question]
--- 
[Answer in simple, clear text.]
---
[Citations in text]
**Source**:
• Page(s): [...]

'''
"""
print("Prompt constructed.") 



Prompt constructed.


In [59]:
print(prompt)


## System role

I am an analytical assistant chatbot, in charge of providing context to the user's question based on the textbook provided. 


The user asks: 
"Mention three main aspects of how AI can influence current political systems."


Here are the most relevant ideas from the authors: 
'''


Content:
humans as workers, the idea of a universal basic income will be put back on the 
table. This would trigger a series of fundamental philosophical but also reli -
gious debates: questions such as the purpose of life, how to feel useful and what 
to strive for, are some issues for which society would have to find answers. Eth-
ics and education will play an important role in order to tackle these societal 
challenges and questions (Kaplan 2020a). 
About Democracy and Freedom
Finally, AI progress could represent nothing less than a danger to peace and 
democracy (Kaplan 2020b). There are at least two ways in which artificial  
intelligence might constitute a threat to democracy and its 

In [37]:
import openai

### Exercice3: Tune parameters like temperature, and penalties to control how creative, focused, or varied the model's responses are.

In [43]:
# Set up GPT client and parameters
client = openai.OpenAI()
model_params = {
    'model': 'gpt-4o-mini',
    'temperature': 0.8,  # Increase creativity
    'max_tokens': 2000,  # Allow for longer responses
    'top_p': 0.9,        # Use nucleus sampling
    'frequency_penalty': 0.5,  # Reduce repetition
    'presence_penalty':  0.7  # Encourage new topics
}

<h1 style="color: #FF6347;">Response</h1>


In [60]:
messages = [{'role': 'user', 'content': prompt}]
completion = client.chat.completions.create(messages=messages, **model_params, timeout=120)

In [61]:
answer = completion.choices[0].message.content
print(answer)

**Summarize user question**  
What are three main ways AI can affect today's political systems?  
---  
AI can influence current political systems in several significant ways:

1. **Impact on Democracy and Freedom**: AI poses risks to democratic processes by enabling manipulation of information. For instance, during the 2016 US presidential election, misinformation spread via social media platforms outperformed reliable news sources in engagement, highlighting how AI-driven systems can shape public opinion and electoral outcomes.

2. **Surveillance and Control**: In countries like China, AI is utilized for extensive surveillance of citizens through technologies such as facial recognition and social credit scoring. This allows governments to monitor behavior closely and enforce compliance, which raises concerns about individual freedoms and privacy rights.

3. **Economic Disparities and Labor Dynamics**: The integration of AI into the workforce could exacerbate existing inequalities by 

<h2 style="color: #FF6347;">Keyword Highlighting</h2>

Highlighting important keywords helps users quickly understand the relevance of the retrieved text to their query.

In [46]:
from termcolor import colored

The `highlight_keywords` function is designed to highlight specific keywords within a given text. It replaces each keyword in the text with a highlighted version using the `colored` function from the `termcolor` library.


In [55]:
def highlight_keywords(text, keywords):
    for keyword in keywords:
        text = text.replace(keyword, colored(keyword, 'green', attrs=['bold']))
    return text

### Exercice4: add your keywords

In [57]:
query_keywords = ["AI", "democracy", "freedom", "power"] # add your keywords
for i, doc in enumerate(retrieved_docs[:1]):
    snippet = doc.page_content[:1000]
    highlighted = highlight_keywords(snippet, query_keywords)
    print(f"Snippet {i+1}:\n{highlighted}\n{'-'*80}")

Snippet 1:
humans as workers, the idea of a universal basic income will be put back on the 
table. This would trigger a series of fundamental philosophical but also reli -
gious debates: questions such as the purpose of life, how to feel useful and what 
to strive for, are some issues for which society would have to find answers. Eth-
ics and education will play an important role in order to tackle these societal 
challenges and questions (Kaplan 2020a). 
About Democracy and Freedom
Finally, AI progress could represent nothing less than a danger to peace and 
democracy (Kaplan 2020b). There are at least two ways in which artificial  
intelligence might constitute a threat to democracy and its mechanisms,  
endangering the peaceful coexistence of humans and machines: supervision 
and manipulation.
Using the example of China, we will provide an illustration as to how far the 
possibilities of artificial intelligence reach with respect to control and super -
------------------------------

1. `query_keywords` is a list of keywords to be highlighted.
2. The loop iterates over the first document in retrieved_docs.
3. For each document, a snippet of the first 200 characters is extracted.
4. The highlight_keywords function is called to highlight the keywords in the snippet.
5. The highlighted snippet is printed along with a separator line.

<h1 style="color: #FF6347;">Bonus</h1>

**Try loading one of your own PDF books and go through the steps again to explore how the pipeline works with your content**:


In [63]:
## Loading document 
article_path = "../Timoshenko2020.pdf"

In [64]:
# Load and split the document
loader2 = PyPDFLoader(article_path)
pages2 = loader2.load_and_split()
len(pages2)

171

In [65]:
text_splitter2 = RecursiveCharacterTextSplitter(
    chunk_size=700,
    chunk_overlap=100
)
chunks2 = text_splitter2.split_documents(pages2)

len(chunks2)

928

In [66]:
db2 = Chroma.from_documents(chunks2, embeddings, persist_directory="chromadb_bonus/chroma_db_bonus")


In [71]:
# Asking questions 
my_question = "Mention at least four examples of Fe K-edge XANES in the text."
retrieved_docs2 = db2.similarity_search(my_question, k=10)

In [72]:
# Display top results
for i, doc in enumerate(retrieved_docs2[:5]): # Display top 3 results
    print(f"Document {i+1}:\n{doc.page_content[0:1000]}") # Display content

Document 1:
flow cell. J. Synchrotron Radiat. 2018, 25, 537 −542.
(392) Braun, A.; Wang, H.; Bergmann, U.; Tucker, M. C.; Gu, W.;
Cramer, S. P.; Cairns, E. J. Origin of chemical shift of manganese in
lithium battery electrode materials a comparison of hard and soft X-
ray techniques. J. Power Sources 2002, 112, 231 −235.
(393) Frenkel, A. I.; Kolobov, A. V.; Robinson, I. K.; Cross, J. O.;
Maeda, Y.; Bouldin, C. E. Direct separation of short range order in
intermixed nanocrystalline and amorphous phases. Phys. Rev. Lett.
2002, 89, 285503.
(394) Frenkel, A. I.; Cross, J.; Fanning, D.; Robinson, I. DAFS
analysis of magnetite. J. Synchrotron Radiat. 1999, 6, 332 −334.
Document 2:
and are also quite sensitive to the oxidation state.
26 For
example, Cu K-edge XANES has no pre-edge feature for Cu(I)
species due to a completely ﬁlled 3d orbital (d
10 state).248 For
the Cu(II) state, there is a single hole in the 3d orbital, and a
pre-edge feature can be observed at ∼8980 eV, Figure 1 a,
aﬀect

In [ ]:
def _get_document_prompt(docs):
    prompt = "\n"
    for doc in docs:
        prompt += "\nContent:\n"
        prompt += doc.page_content + "\n\n"
    return prompt

In [73]:
# Generate a formatted context from the retrieved documents
formatted_context2 = _get_document_prompt(retrieved_docs2)
print("Context formatted for GPT model.")

Context formatted for GPT model.


In [76]:
prompt2 = f"""
## System role

I am an analytical assistant chatbot, in charge of providing context to the user's question based on the textbook provided. 


The user asks: 
"{my_question}"


Here are the most relevant ideas from the authors: 
'''
{formatted_context2}

'''
## GUIDELINES: 
1. **Accuracy** 
- Use only the content provided to answer the question. 
- When the answer is not found, state: "The provided context does not contain information related to the user's question."
2. **Transparency**:
   - Reference the book's chapter title, author and page numbers when providing information.
   - Do not speculate or provide opinions.

3. **Clarity**:
   - Use simple, professional, and concise language.
   - Format your response in Markdown for readability.

## TASK
1. Summarize user question in simpler words. 
2. Decide which retrieved text chunks directly apply. .
3. Combine those chunks into an outline.
4. Draft a single coherent answer. 
5. Show the most relevant quotes from the authors, each with the page number where it can be found. 

## RESPONSE FORMAT
'''
[Summarize user question]
--- 
[Answer in simple, clear text.]
---
[Citations in text]
**Source**:
• Page(s): [...]

'''
"""
print("Prompt constructed.") 



Prompt constructed.


In [77]:
print(prompt2)


## System role

I am an analytical assistant chatbot, in charge of providing context to the user's question based on the textbook provided. 


The user asks: 
"Mention at least four examples of Fe K-edge XANES in the text."


Here are the most relevant ideas from the authors: 
'''


Content:
flow cell. J. Synchrotron Radiat. 2018, 25, 537 −542.
(392) Braun, A.; Wang, H.; Bergmann, U.; Tucker, M. C.; Gu, W.;
Cramer, S. P.; Cairns, E. J. Origin of chemical shift of manganese in
lithium battery electrode materials a comparison of hard and soft X-
ray techniques. J. Power Sources 2002, 112, 231 −235.
(393) Frenkel, A. I.; Kolobov, A. V.; Robinson, I. K.; Cross, J. O.;
Maeda, Y.; Bouldin, C. E. Direct separation of short range order in
intermixed nanocrystalline and amorphous phases. Phys. Rev. Lett.
2002, 89, 285503.
(394) Frenkel, A. I.; Cross, J.; Fanning, D.; Robinson, I. DAFS
analysis of magnetite. J. Synchrotron Radiat. 1999, 6, 332 −334.


Content:
and are also quite sensitive to t

In [81]:
messages2 = [{'role': 'user', 'content': prompt2}]
completion2 = client.chat.completions.create(messages=messages2, **model_params, timeout=120)

In [82]:
answer2 = completion2.choices[0].message.content
print(answer2)

'''
The user wants to know about four examples of Fe K-edge XANES mentioned in the text. 
--- 
The provided context includes discussions on the use of Fe K-edge XANES for various applications, but it does not specify four distinct examples. However, there are references that highlight its significance in different studies:

1. The analysis of pre-edge features in Fe K-edge spectra helps discriminate between Fe(II) and Fe(III) species, which is particularly useful in environmental science.
2. Zitolo et al. demonstrated that XANES data fitting of iron-based materials provides important insights during the oxygen reduction reaction (ORR).
3. Wilke et al. conducted a spectroscopic study using Fe K-XANES to examine the oxidation state and coordination of iron in minerals.
4. Friebel et al. utilized Fe K-edge measurements to detect changes in structure and oxidation state under electrochemical conditions.

These references illustrate how Fe K-edge XANES is applied across different fields and